# LABORATORIO DIRIGIDO N.° 03
**Semana 3 · SQL Avanzado — Joins y Agrupaciones**  
Clientes, pedidos y productos de TecnoAndina Retail con JOIN, agrupaciones y vistas en SQLite  
Curso: Fundamentos de Gestión de Datos · Periodo: 2026-II · Docente: Pilar Rocío Sayán Mejía  
Datos: `LD3_TecnoAndina_Retail.xlsx` (GitHub: Rociosayan/Repo_FDG_2026-2, carpeta semana03/datos/)

## Caso de estudio: TecnoAndina Retail

TecnoAndina Retail es la cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web, con la que trabajó en la Semana 2. Allí analizó una sola tabla de ventas. Ahora el área comercial dispone de una base relacional de pedidos, en la que la información está repartida en cuatro tablas: los clientes, sus pedidos, el catálogo de productos y el detalle de lo que contiene cada pedido. Usted continúa como analista junior del área comercial.

El problema aparece en la reunión de seguimiento. La jefatura quiere planificar una campaña de fidelización y reactivación, pero cada dato necesario está en una tabla distinta: el nombre y la ciudad del cliente no están en el pedido, y el producto vendido no está en el cliente. Nadie sabe cuántos clientes registrados no han comprado, qué productos se venden más en unidades frente a los que dejan más dinero, ni qué clientes han acumulado más compras. La pregunta de negocio es: «¿Qué clientes han acumulado mayor monto de compras, cuáles todavía no compran y qué productos y categorías sustentan esas ventas?». El encargo no consiste en predecir ni en explicar causas, sino en responder con consultas SQL sobre los pedidos registrados y señalar qué información adicional haría falta para decidir.

Naturaleza de los datos.  Esta base es un conjunto de práctica distinto del de la Semana 2: no contiene los 72 registros de ventas de ese laboratorio y sus cifras no deben mezclarse ni compararse con ellas. Los datos son de uso exclusivamente académico: no corresponden a una empresa real ni a personas reales. Los pedidos cubren del 1 de agosto al 30 de septiembre de 2026 (dos meses), y están publicados en un archivo Excel fijo en GitHub, de modo que todos trabajan con los mismos registros. Por su tamaño y periodo, sirven para practicar consultas, no para afirmar tendencias, rentabilidad ni causas del negocio completo.

### Lo que se le pide:

1.  Reconocer las cuatro tablas, sus claves y comprobar que la base está completa y sin relaciones inválidas.

2.  Relacionar clientes, pedidos y productos con INNER JOIN, LEFT JOIN, RIGHT JOIN y FULL OUTER JOIN.

3.  Agrupar con GROUP BY y filtrar grupos con HAVING, distinguiéndolos de WHERE y tratando los NULL.

4.  Reutilizar resultados con subconsultas y vistas, definiendo con claridad qué se compara.

5.  Elaborar el informe de los cinco clientes con mayor monto acumulado y una recomendación prudente.

### Qué significa cada columna de las cuatro tablas:

| Tabla | Columna | Significado | Tipo | Clave |
|---|---|---|---|---|
| clientes | id_cliente | Número que identifica a cada cliente. | INTEGER | Clave primaria |
| clientes | nombre | Nombre del cliente (dos clientes distintos pueden tener el mismo nombre). | TEXT |  |
| clientes | ciudad | Ciudad del cliente: Lima, Arequipa, Trujillo o Piura. | TEXT |  |
| productos | id_producto | Número que identifica a cada producto del catálogo. | INTEGER | Clave primaria |
| productos | nombre_producto | Nombre del producto; hay catorce. | TEXT |  |
| productos | categoria | Grupo del producto: Cómputo, Celulares o Accesorios. | TEXT |  |
| productos | precio_lista | Precio de catálogo, en soles. | REAL |  |
| pedidos | id_pedido | Número que identifica cada pedido. | INTEGER | Clave primaria |
| pedidos | id_cliente | Cliente que hizo el pedido. | INTEGER | Clave foránea → clientes |
| pedidos | fecha_pedido | Día del pedido, en formato AAAA-MM-DD. | TEXT |  |
| pedidos | canal | Tienda (presencial) o Web. | TEXT |  |
| pedidos | total | Monto total del pedido, en soles; es la suma de sus detalles. | REAL |  |
| detalles_pedido | id_detalle | Número que identifica cada línea del pedido. | INTEGER | Clave primaria |
| detalles_pedido | id_pedido | Pedido al que pertenece la línea. | INTEGER | Clave foránea → pedidos |
| detalles_pedido | id_producto | Producto vendido en la línea. | INTEGER | Clave foránea → productos |
| detalles_pedido | cantidad | Unidades vendidas en la línea. | INTEGER |  |
| detalles_pedido | precio_unitario | Precio cobrado por unidad, en soles (puede ser menor al de lista por promoción). | REAL |  |

### Cómo se relacionan las tablas

- Un cliente puede tener varios pedidos, y cada pedido pertenece a un solo cliente. Se enlazan por id_cliente: en clientes es la clave primaria y en pedidos es la clave foránea.

- Un pedido puede incluir varias líneas de detalle, y cada línea pertenece a un solo pedido. Se enlazan por id_pedido.

- Un producto puede aparecer en muchas líneas de detalle, y cada línea se refiere a un solo producto. Se enlazan por id_producto.

- Por eso detalles_pedido une a pedidos con productos: un pedido contiene varios productos y un producto se vende en varios pedidos.

- En esta base hay clientes que aún no tienen pedidos y productos que aún no se han vendido; ninguna fila de pedidos o de detalles apunta a un cliente, pedido o producto inexistente.

### Preparación del entorno

Abra el notebook en Google Colab y ejecute la primera celda. Esta importa sqlite3 y pandas, crea la base en memoria, define las cuatro tablas con sus claves primarias y foráneas, lee cada hoja del archivo Excel publicado en GitHub, la agrega a su tabla con to_sql y confirma los cambios con commit. Cada consulta se muestra con pd.read_sql_query(consulta, conexion); cuando una celda presenta dos resultados, se usa display(). En esas tablas, pandas muestra los NULL de SQL como NaN o None. No modifique esta celda; el código se reproduce aquí para seguir el procedimiento aun sin abrir el notebook. Si reinicia el entorno, vuelva a ejecutar la celda 1 y las celdas de las que dependa (por ejemplo, la 12 y la 23 crean vistas).

**Celda 1 de 30**

In [ ]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail: pedidos del 1 de agosto al
# 30 de septiembre de 2026. Los datos están en un Excel publicado en GitHub.
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"

conexion = sqlite3.connect(":memory:")
conexion.execute("PRAGMA foreign_keys = ON")  # SQLite exige activarlo

# 1) Estructura: las cuatro tablas con sus claves primarias y foráneas
conexion.executescript("""
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    ciudad TEXT NOT NULL
);
CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    nombre_producto TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio_lista REAL NOT NULL
);
CREATE TABLE pedidos (
    id_pedido INTEGER PRIMARY KEY,
    id_cliente INTEGER NOT NULL,
    fecha_pedido TEXT NOT NULL,
    canal TEXT NOT NULL,
    total REAL NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes (id_cliente)
);
CREATE TABLE detalles_pedido (
    id_detalle INTEGER PRIMARY KEY,
    id_pedido INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    FOREIGN KEY (id_pedido) REFERENCES pedidos (id_pedido),
    FOREIGN KEY (id_producto) REFERENCES productos (id_producto)
);
""")

# 2) Datos: cada hoja del Excel se agrega a su tabla
clientes = pd.read_excel(url, sheet_name="clientes")
productos = pd.read_excel(url, sheet_name="productos")
pedidos = pd.read_excel(url, sheet_name="pedidos")
detalles_pedido = pd.read_excel(url, sheet_name="detalles_pedido")

clientes.to_sql("clientes", conexion, if_exists="append", index=False)
productos.to_sql("productos", conexion, if_exists="append", index=False)
pedidos.to_sql("pedidos", conexion, if_exists="append", index=False)
detalles_pedido.to_sql("detalles_pedido", conexion, if_exists="append", index=False)
conexion.commit()

print("Base preparada: 4 tablas creadas y cargadas desde GitHub.")

## A. Relacionar tablas

### Ejercicio 1 — Reconocer las tablas y comprobar su integridad

Antes de cruzar tablas, la jefatura exige confirmar que la base está completa y que ningún pedido apunta a un cliente o producto inexistente. El analista revisa qué objetos hay, cuántos registros tiene cada tabla, cómo se enlazan y si los totales cuadran.

Se solicita. Ejecute las tres celdas: la primera lista los objetos y cuenta los registros, la segunda muestra las claves foráneas y la tercera verifica la integridad con consultas cuyo resultado esperado es 0.

**Celda 2 de 30**

In [ ]:
consulta = """SELECT name AS objeto, type AS tipo
FROM sqlite_master
WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%'
ORDER BY name;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT
    (SELECT COUNT(*) FROM clientes) AS clientes,
    (SELECT COUNT(*) FROM productos) AS productos,
    (SELECT COUNT(*) FROM pedidos) AS pedidos,
    (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
display(pd.read_sql_query(consulta, conexion))

**Celda 3 de 30**

In [ ]:
consulta = """SELECT 'pedidos' AS tabla, "from" AS columna,
       "table" AS tabla_referida, "to" AS columna_referida
FROM pragma_foreign_key_list('pedidos')
UNION ALL
SELECT 'detalles_pedido', "from", "table", "to"
FROM pragma_foreign_key_list('detalles_pedido');"""
pd.read_sql_query(consulta, conexion)

**Celda 4 de 30**

In [ ]:
consulta = """SELECT
  (SELECT COUNT(*)
   FROM pedidos AS p
   LEFT JOIN clientes AS c ON p.id_cliente = c.id_cliente
   WHERE c.id_cliente IS NULL) AS pedidos_sin_cliente,
  (SELECT COUNT(*)
   FROM detalles_pedido AS d
   LEFT JOIN pedidos AS p ON d.id_pedido = p.id_pedido
   WHERE p.id_pedido IS NULL) AS detalles_sin_pedido,
  (SELECT COUNT(*)
   FROM detalles_pedido AS d
   LEFT JOIN productos AS pr ON d.id_producto = pr.id_producto
   WHERE pr.id_producto IS NULL) AS detalles_sin_producto,
  (SELECT COUNT(*)
   FROM pedidos AS p
   INNER JOIN (SELECT id_pedido, SUM(cantidad * precio_unitario) AS suma
               FROM detalles_pedido
               GROUP BY id_pedido) AS d ON p.id_pedido = d.id_pedido
   WHERE p.total <> d.suma) AS totales_que_no_cuadran;"""
display(pd.read_sql_query(consulta, conexion))

# Revisión de claves foráneas: una tabla vacía significa que no hay violaciones
display(pd.read_sql_query("PRAGMA foreign_key_check;", conexion))

**Pregunta 1.** ¿Cuántos registros tiene cada tabla? ¿Cuáles de las cuatro tablas no dependen de ninguna otra mediante una clave foránea y cuáles sí?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 2.** Complete con las columnas que muestra la celda 3: pedidos se enlaza con clientes mediante ______; detalles_pedido enlaza con pedidos mediante ______ y con productos mediante ______.

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 3.** En la celda 4, ¿qué significa que las cuatro verificaciones den 0? ¿Qué problema habría en los informes si pedidos_sin_cliente fuera 2?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 2 — INNER JOIN: pedidos con los datos de su cliente

El área comercial quiere un listado de pedidos con el nombre y la ciudad de quien compró. Esa información está repartida: el pedido guarda solo id_cliente y el nombre está en clientes. Un INNER JOIN conserva únicamente las filas que encuentran pareja en ambas tablas.

Se solicita. Ejecute las dos celdas: la primera muestra los diez pedidos más recientes con los datos del cliente; la segunda cuenta, sobre todos los pedidos, cuántos clientes distintos resultan del cruce.

**Celda 5 de 30**

In [ ]:
consulta = """SELECT p.id_pedido, p.fecha_pedido,
       c.id_cliente, c.nombre, c.ciudad, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
ORDER BY p.fecha_pedido DESC, p.id_pedido DESC
LIMIT 10;"""
pd.read_sql_query(consulta, conexion)

**Celda 6 de 30**

In [ ]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 4.** ¿Qué columna de cada tabla compara la condición ON? ¿Qué representan los alias p y c y por qué ayudan en esta consulta?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 5.** La celda 5 muestra solo diez filas. ¿Qué cláusula las limita y por qué su resultado no sirve para contar cuántos clientes tienen pedidos? ¿Qué cifras entrega la celda 6?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 6.** Compare clientes_con_pedidos con clientes_registrados. ¿Qué diferencia hay y por qué un INNER JOIN no permite ver a esos clientes?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 3 — LEFT JOIN: conservar a los clientes sin pedidos

Marketing quiere reactivar a los clientes que todavía no compran. Para no perderlos, la consulta debe partir de la tabla clientes y conservar todas sus filas, tengan o no pedidos. Cuando no hay pareja, las columnas del pedido quedan en NULL: un valor ausente, distinto de cero.

Se solicita. Ejecute las dos celdas: la primera muestra los clientes de Piura con sus pedidos, para observar los NULL; la segunda lista y cuenta, en toda la base, a los clientes sin pedidos.

**Celda 7 de 30**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       p.id_pedido, p.fecha_pedido, p.total
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE c.ciudad = 'Piura'
ORDER BY c.id_cliente, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

**Celda 8 de 30**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL;"""
display(pd.read_sql_query(consulta, conexion))

**Pregunta 7.** En la celda 7, ¿qué valor aparece en id_pedido y total para los clientes de Piura sin compras? ¿Qué significa ese valor y en qué se diferencia de un total igual a 0?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 8.** ¿Por qué la celda 8 filtra con p.id_pedido IS NULL y no con p.id_pedido = 0 ni con p.id_pedido = NULL? ¿Por qué se revisa una columna de pedidos y no una de clientes?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 9.** Anote cuántos clientes no tienen pedidos y verifique si coincide con la diferencia entre clientes_registrados y clientes_con_pedidos de la celda 6.

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 10.** Si la consulta de la celda 8 se escribiera como FROM pedidos AS p LEFT JOIN clientes AS c, ¿aparecerían los clientes sin pedidos? Explique qué tabla se conserva completa en cada caso.

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 4 — RIGHT JOIN: versión de SQLite y alternativa equivalente

El área de compras pregunta qué productos del catálogo no se vendieron entre agosto y septiembre. Se parte de los detalles de pedido y se desea conservar todos los productos, que quedan a la derecha del JOIN. RIGHT JOIN y FULL OUTER JOIN existen en SQLite solo desde la versión 3.39.0 (2022); en un entorno más antiguo, la consulta falla. Por eso se comprueba primero la versión. Además, todo RIGHT JOIN puede escribirse como LEFT JOIN intercambiando el orden de las tablas.

Se solicita. Ejecute las tres celdas: la primera informa la versión de SQLite; la segunda usa RIGHT JOIN (requiere la versión 3.39.0 o posterior) y la tercera obtiene lo mismo con LEFT JOIN.

**Celda 9 de 30**

In [ ]:
# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
pd.read_sql_query(consulta, conexion)

**Celda 10 de 30**

In [ ]:
consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
pd.read_sql_query(consulta, conexion)

**Celda 11 de 30**

In [ ]:
consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM productos AS pr
LEFT JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 11.** ¿Qué versión de SQLite informó la celda 9 y admite RIGHT JOIN? Si no lo admitiera, ¿qué celda reemplazaría a la celda 10?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 12.** En la celda 10, ¿qué tabla es la «derecha» y qué condición del WHERE deja solo los productos sin ventas? ¿Cuáles son esos productos?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 13.** Explique con sus palabras por qué la celda 11 equivale a la celda 10 aunque cambien el orden de las tablas y la palabra LEFT. ¿Coinciden las filas que devuelven ambas celdas?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 5 — FULL OUTER JOIN: comparar dos periodos por cliente

La jefatura quiere comparar a cada cliente entre agosto y septiembre. Hay clientes que compraron solo en uno de los dos meses y ningún resumen debe perderlos. FULL OUTER JOIN conserva las filas de ambos lados, con o sin pareja. Para que el cruce sea claro, primero se crean dos resúmenes por cliente, uno por mes, como vistas temporales (existen solo mientras dura la conexión).

Se solicita. Ejecute las tres celdas: la primera crea los dos resúmenes; la segunda los cruza con FULL OUTER JOIN (requiere la versión 3.39.0 o posterior) y la tercera obtiene lo mismo con LEFT JOIN y UNION ALL, que apila los resultados de dos consultas con las mismas columnas.

**Celda 12 de 30**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT (SELECT COUNT(*) FROM resumen_agosto) AS clientes_agosto,
       (SELECT COUNT(*) FROM resumen_septiembre) AS clientes_septiembre;"""
pd.read_sql_query(consulta, conexion)

**Celda 13 de 30**

In [ ]:
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.num_pedidos AS pedidos_agosto, a.monto AS monto_agosto,
       s.num_pedidos AS pedidos_septiembre, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Celda 14 de 30**

In [ ]:
consulta = """SELECT a.id_cliente,
       a.num_pedidos AS pedidos_agosto, a.monto AS monto_agosto,
       s.num_pedidos AS pedidos_septiembre, s.monto AS monto_septiembre
FROM resumen_agosto AS a
LEFT JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
UNION ALL
SELECT s.id_cliente, NULL, NULL, s.num_pedidos, s.monto
FROM resumen_septiembre AS s
LEFT JOIN resumen_agosto AS a ON s.id_cliente = a.id_cliente
WHERE a.id_cliente IS NULL
ORDER BY 1;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 14.** ¿Cuántos clientes aparecen solo en agosto, cuántos solo en septiembre y cuántos en ambos meses? Indique cómo los reconoció en el resultado de la celda 13.

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 15.** Los clientes sin pedidos en toda la base, ¿aparecen en este resultado? ¿Por qué? ¿Qué tendría que cambiar en el planteamiento para incluirlos?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 16.** Explique cómo la celda 14 reproduce el FULL OUTER JOIN: qué parte conserva lo de agosto, qué parte agrega lo exclusivo de septiembre y por qué el WHERE a.id_cliente IS NULL evita duplicados.

**Respuesta:** _Escriba aquí su respuesta._

## B. Agrupar y filtrar grupos

### Ejercicio 6 — Resumen por cliente: contar, sumar y tratar los NULL

Ventas necesita una fila por cliente con cuántos pedidos hizo y cuánto compró, incluidos quienes no compraron. Si un cliente no tiene pedidos, SUM y AVG devuelven NULL. COALESCE reemplaza un NULL por otro valor, pero solo conviene donde ese valor tenga sentido: un monto acumulado sin compras es 0; un promedio por pedido sin pedidos no existe.

Se solicita. Ejecute la celda y compare las columnas filas_del_join y num_pedidos, y el tratamiento de monto_acumulado y promedio_pedido.

**Celda 15 de 30**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(*) AS filas_del_join,                     -- incluye la fila vacía
       COUNT(p.id_pedido) AS num_pedidos,              -- solo pedidos reales
       COALESCE(SUM(p.total), 0) AS monto_acumulado,   -- sin compras: 0
       ROUND(AVG(p.total), 2) AS promedio_pedido       -- sin compras: NULL
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 17.** Compare filas_del_join y num_pedidos en un cliente sin pedidos. ¿Qué cuenta cada columna y por qué para contar pedidos se usa COUNT(p.id_pedido) y no COUNT(*)?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 18.** ¿Por qué monto_acumulado muestra 0 y promedio_pedido muestra NULL en esos clientes? ¿Qué error de interpretación causaría convertir ese NULL en un promedio de 0?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 19.** ¿Por qué el GROUP BY incluye id_cliente, nombre y ciudad, y no solo nombre? Señale dos clientes del resultado que comparten nombre y cómo se distinguen.

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 7 — WHERE y HAVING: filtrar filas y filtrar grupos

La jefatura formula tres pedidos distintos de información. WHERE filtra filas antes de agrupar; HAVING filtra los grupos ya formados, usando resultados de funciones de agregación. Ambas cláusulas pueden aparecer en la misma consulta junto con GROUP BY, cada una con su función.

Se solicita. Ejecute las tres consultas de la celda: clientes de Lima y su monto (WHERE con GROUP BY); clientes con tres o más pedidos (HAVING); y clientes con al menos dos pedidos en septiembre (WHERE y HAVING combinados).

**Celda 16 de 30**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS num_pedidos, SUM(p.total) AS monto_acumulado
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE c.ciudad = 'Lima'
GROUP BY c.id_cliente, c.nombre
ORDER BY monto_acumulado DESC, c.id_cliente;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos, SUM(p.total) AS monto_acumulado
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
HAVING COUNT(p.id_pedido) >= 3
ORDER BY num_pedidos DESC, c.id_cliente;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre, SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
display(pd.read_sql_query(consulta, conexion))

**Pregunta 20.** En la primera consulta, ¿qué hace WHERE c.ciudad = 'Lima' y en qué momento actúa respecto de GROUP BY? ¿Por qué WHERE y GROUP BY pueden convivir en la misma consulta?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 21.** En la segunda consulta, ¿por qué la condición de tres o más pedidos no puede escribirse en WHERE? ¿Qué clientes la cumplen?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 22.** En la tercera consulta, indique qué filtra WHERE (filas) y qué filtra HAVING (grupos). Un cliente con tres pedidos en total, pero uno solo en septiembre, ¿aparece en el resultado? Justifique.

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 8 — Productos más vendidos y productos que generan mayor ingreso

Compras pide dos rankings que no son lo mismo: qué productos salen más (unidades) y cuáles dejan más dinero (ingreso). Un producto barato puede venderse mucho y aportar poco. El ingreso se calcula desde cantidad × precio_unitario de cada línea de detalle. Cuando dos productos empatan, un criterio de desempate (aquí, id_producto) fija un orden estable.

Se solicita. Ejecute la celda: la primera consulta ordena por unidades y la segunda por ingreso; ambas muestran las dos medidas para poder compararlas.

**Celda 17 de 30**

In [ ]:
# Productos más vendidos (ordenados por unidades)
consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria,
       SUM(d.cantidad) AS unidades_vendidas,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.id_producto, pr.nombre_producto, pr.categoria
ORDER BY unidades_vendidas DESC, pr.id_producto
LIMIT 5;"""
display(pd.read_sql_query(consulta, conexion))

# Productos que generan mayor ingreso (ordenados por ingreso)
consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria,
       SUM(d.cantidad) AS unidades_vendidas,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.id_producto, pr.nombre_producto, pr.categoria
ORDER BY ingreso DESC, pr.id_producto
LIMIT 5;"""
display(pd.read_sql_query(consulta, conexion))

**Pregunta 23.** ¿Qué dos productos empatan en unidades dentro del primer ranking? ¿Qué criterio de desempate decidió su orden y por qué es necesario escribirlo?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 24.** Compare ambas listas: ¿qué productos aparecen en una y no en la otra? ¿Qué mide cada ranking?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 25.** Un gerente lee la segunda lista y dice «estos son los productos más vendidos». ¿Es correcto? Redacte un título adecuado para cada consulta.

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 9 — Evitar duplicar totales al unir pedidos con sus detalles

Un practicante calcula el ingreso sumando pedidos.total después de unir cada pedido con sus detalles y obtiene una cifra mayor que la de contabilidad. El total de un pedido se repite en cada una de sus líneas de detalle, de modo que al unir ambas tablas se cuenta más de una vez.

Se solicita. Ejecute la celda y compare las tres cifras: ingreso desde pedidos, ingreso desde detalles (cantidad × precio_unitario) y la suma duplicada.

**Celda 18 de 30**

In [ ]:
consulta = """SELECT
  (SELECT SUM(total) FROM pedidos) AS ingreso_desde_pedidos,
  (SELECT SUM(cantidad * precio_unitario)
   FROM detalles_pedido) AS ingreso_desde_detalles,
  (SELECT SUM(p.total)
   FROM pedidos AS p
   INNER JOIN detalles_pedido AS d ON p.id_pedido = d.id_pedido) AS suma_duplicada;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 26.** ¿Qué dos cifras coinciden y cuál es incorrecta? ¿Cuál es la diferencia entre la incorrecta y la correcta?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 27.** Explique por qué se infla la tercera cifra: ¿cuántas filas genera al unirse un pedido con tres líneas de detalle? ¿Qué debe sumarse para obtener ingresos por producto sin duplicar?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 10 — Ventas por ciudad del cliente

La gerencia regional pide, por ciudad del cliente, cuántos clientes están registrados, cuántos compraron, cuántos pedidos hubo y qué monto se acumuló. La consulta parte de clientes con LEFT JOIN para que las ciudades con clientes sin compras no pierdan información.

Se solicita. Ejecute la celda y observe qué cuenta cada columna.

**Celda 19 de 30**

In [ ]:
consulta = """SELECT c.ciudad,
       COUNT(DISTINCT c.id_cliente) AS clientes_registrados,
       COUNT(DISTINCT p.id_cliente) AS clientes_con_pedidos,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_total
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.ciudad
ORDER BY monto_total DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 28.** ¿Qué ciudad acumula mayor monto? ¿Cuál tiene proporción más alta de clientes sin pedidos? Cite las cifras.

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 29.** ¿Por qué se usa COUNT(DISTINCT c.id_cliente) para los clientes y COUNT(p.id_pedido) para los pedidos? ¿Qué cifra incorrecta daría COUNT(c.id_cliente) para los clientes?

**Respuesta:** _Escriba aquí su respuesta._

## C. Consultar y reutilizar resultados

### Ejercicio 11 — Subconsulta: compras acumuladas frente al promedio por cliente

Comercial quiere identificar a los clientes cuyas compras acumuladas superan el promedio de compras acumuladas por cliente. Antes de consultar hay que definir qué es «por cliente»: solo quienes compraron o los 24 clientes registrados. Una subconsulta calcula primero un valor o una tabla intermedia que la consulta principal utiliza después.

Se solicita. Ejecute las dos celdas: la primera calcula el promedio de compras acumuladas con las dos definiciones; la segunda lista a los clientes con pedidos cuyo monto acumulado supera el promedio calculado solo entre quienes compraron.

**Celda 20 de 30**

In [ ]:
consulta = """SELECT
  ROUND(AVG(monto_cliente), 2) AS promedio_solo_con_pedidos,
  ROUND(SUM(monto_cliente) * 1.0 / (SELECT COUNT(*) FROM clientes), 2)
      AS promedio_todos_los_clientes
FROM (SELECT id_cliente, SUM(total) AS monto_cliente
      FROM pedidos
      GROUP BY id_cliente) AS compras;"""
pd.read_sql_query(consulta, conexion)

**Celda 21 de 30**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
HAVING SUM(p.total) > (SELECT AVG(monto_cliente)
                       FROM (SELECT SUM(total) AS monto_cliente
                             FROM pedidos
                             GROUP BY id_cliente) AS compras)
ORDER BY monto_acumulado DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 30.** ¿Qué diferencia numérica hay entre los dos promedios de la celda 20 y a qué se debe? ¿Cuál usa la celda 21? Justifíquelo con la pregunta comercial.

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 31.** Identifique en la celda 21 la subconsulta más interna y diga qué devuelve (cuántas filas y qué columnas). ¿Qué devuelve la subconsulta que rodea a AVG? ¿Por qué se compara en HAVING y no en WHERE?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 32.** ¿Cuántos clientes superan el promedio? Compare sus montos con los dos promedios de la celda 20: ¿cambiaría la lista si se usara el otro promedio?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 12 — Subconsulta: pedidos individuales frente al promedio por pedido

Logística quiere revisar los pedidos grandes, es decir, los que superan el monto de un pedido promedio. Aquí la unidad de análisis es el pedido, no el cliente: se compara el total de cada pedido con el promedio de totales por pedido. Es otra pregunta, con otra medida, distinta de la del ejercicio 11.

Se solicita. Ejecute la celda y compruebe que cada fila supera el promedio por pedido que se muestra a su derecha.

**Celda 22 de 30**

In [ ]:
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total,
       (SELECT ROUND(AVG(total), 2) FROM pedidos) AS promedio_por_pedido
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 33.** ¿Qué medida compara la celda 21 y cuál la celda 22? ¿Por qué no deben mezclarse el promedio por cliente y el promedio por pedido?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 34.** ¿Cuántos pedidos superan el promedio por pedido? ¿Qué clientes aparecen más de una vez y qué indica eso?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 13 — Vistas: guardar una consulta para reutilizarla

Tres áreas piden el mismo resumen por cliente con pequeñas variantes. En lugar de copiar la consulta, el analista guarda las consultas base como vistas: una vista es una consulta con nombre, que se vuelve a ejecutar cada vez que se la consulta. Las vistas incluyen los identificadores para distinguir clientes con el mismo nombre.

Se solicita. Ejecute las tres celdas: la primera crea las vistas vista_resumen_cliente y vista_ventas_producto; la segunda consulta la primera vista; la tercera consulta la segunda.

**Celda 23 de 30**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
DROP VIEW IF EXISTS vista_ventas_producto;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
CREATE VIEW vista_ventas_producto AS
SELECT pr.id_producto, pr.nombre_producto, pr.categoria,
       COALESCE(SUM(d.cantidad), 0) AS unidades_vendidas,
       COALESCE(SUM(d.cantidad * d.precio_unitario), 0) AS ingreso
FROM productos AS pr
LEFT JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.id_producto, pr.nombre_producto, pr.categoria;
""")
conexion.commit()
consulta = """SELECT name AS vista
FROM sqlite_master
WHERE type = 'view'
ORDER BY name;"""
pd.read_sql_query(consulta, conexion)

**Celda 24 de 30**

In [ ]:
consulta = """SELECT id_cliente, nombre, ciudad, num_pedidos, monto_acumulado,
       promedio_pedido
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Celda 25 de 30**

In [ ]:
consulta = """SELECT id_producto, nombre_producto, categoria,
       unidades_vendidas, ingreso
FROM vista_ventas_producto
ORDER BY ingreso DESC, id_producto;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 35.** Si mañana se registra un pedido nuevo, ¿cambiará el resultado de vista_resumen_cliente? ¿Qué guarda realmente una vista: los datos o la consulta?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 36.** En la celda 24 aparecen dos clientes llamados Carlos Mendoza. ¿Qué columna de la vista permite distinguirlos y qué pasaría con un informe que solo mostrara el nombre?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 37.** ¿Qué filas aparecen en vista_ventas_producto que no aparecían en los rankings de la celda 17? ¿Por qué la vista usa LEFT JOIN?

**Respuesta:** _Escriba aquí su respuesta._

## Actividad 3 — Informe analítico comparado

La gerencia comercial evalúa ofrecer beneficios a los clientes que más han comprado. Con lo aprendido, elabore el informe de los cinco clientes con mayor monto acumulado: identificador y nombre, ciudad, número de pedidos, monto acumulado y monto promedio por pedido. Estos clientes lideran por monto acumulado en dos meses; eso no los convierte en los más rentables ni en los de mayor valor de vida. Ejecute las dos celdas siguientes, complete la tabla y responda en términos de negocio, citando cifras del notebook.

**Celda 26 de 30**

In [ ]:
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

**Celda 27 de 30**

In [ ]:
consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes_registrados,
  (SELECT COUNT(DISTINCT id_cliente) FROM pedidos) AS clientes_con_pedidos,
  (SELECT COUNT(c.id_cliente)
   FROM clientes AS c
   LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
   WHERE p.id_pedido IS NULL) AS clientes_sin_pedidos;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT SUM(monto_acumulado) AS monto_top_cinco,
  (SELECT SUM(total) FROM pedidos) AS monto_total_base,
  ROUND(100.0 * SUM(monto_acumulado) / (SELECT SUM(total) FROM pedidos), 1)
      AS porcentaje_del_total
FROM (SELECT id_cliente, monto_acumulado
      FROM vista_resumen_cliente
      ORDER BY monto_acumulado DESC, id_cliente
      LIMIT 5) AS top_cinco;"""
display(pd.read_sql_query(consulta, conexion))

| Posición | id_cliente | Nombre | Ciudad | Pedidos | Monto acumulado (S/) | Promedio por pedido (S/) |
|---|---|---|---|---|---|---|
| 1 |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |
| 3 |  |  |  |  |  |  |
| 4 |  |  |  |  |  |  |
| 5 |  |  |  |  |  |  |

**A.** ¿Qué clientes lideran por monto acumulado? Mencione los dos primeros con sus cifras.

**Respuesta:** _Escriba aquí su respuesta._

**B.** ¿Cómo difieren la frecuencia (número de pedidos) y el monto promedio por pedido entre los cinco clientes? Identifique uno de alta frecuencia y otro de monto promedio alto, y explique qué los distingue.

**Respuesta:** _Escriba aquí su respuesta._

**C.** ¿Qué ocurre con los clientes sin pedidos en este informe? ¿Aparecen en el ranking? Use las cifras de la celda 27 para decir cuántos son y qué proporción de la base representan.

**Respuesta:** _Escriba aquí su respuesta._

**D.** ¿Qué información adicional pediría antes de recomendar beneficios a estos clientes? Mencione al menos tres datos que no están en la base.

**Respuesta:** _Escriba aquí su respuesta._

**E.** Redacte en tres líneas una recomendación prudente para la gerencia, con cifras y señalando qué no puede afirmarse con esta base.

**Respuesta:** _Escriba aquí su respuesta._

## Reto de aplicación y retroalimentación

En esta sección se aplican los procedimientos desarrollados durante la sesión a nuevas situaciones del mismo caso. Cada reto presenta la consulta ya escrita sobre la base relacional: el estudiante la ejecuta, lee qué pide cada cláusula e interpreta el resultado desde una perspectiva comercial. El propósito es comprobar la lectura correcta de las consultas y la capacidad de convertir un resultado en una recomendación.

Indicaciones generales. Los retos operan sobre la base creada en la celda 1; el reto 3 usa además las vistas creadas en la celda 23. Cada respuesta escrita no debe exceder cuatro líneas. Los retos se ejecutan e interpretan en el notebook y quedan como evidencia en el repositorio de GitHub.

### Reto 1 — Clientes sin pedidos: oportunidad de reactivación

Marketing prepara un mensaje de reactivación y necesita saber en qué ciudades se concentran los clientes registrados que todavía no tienen pedidos. Un cliente sin pedidos no es necesariamente un cliente perdido: la base cubre solo dos meses y no registra cuándo se dio de alta cada cliente.

Se solicita:

1.  Ejecutar la consulta de la celda, que conserva a todos los clientes con LEFT JOIN, deja solo los que no tienen pedidos y los cuenta por ciudad.

2.  Identificar qué cláusula conserva a los clientes sin pedidos y cuál los aísla.

3.  Interpretar el resultado para decidir a qué ciudades dirigir el mensaje.

**Celda 28 de 30**

In [ ]:
consulta = """SELECT c.ciudad,
       COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

**Pregunta de cierre.** ¿Cuántos clientes sin pedidos hay en total y en qué ciudades se concentran? ¿Qué acción propondría y qué dato adicional necesitaría antes de contactarlos?

**Respuesta:** _Escriba aquí su respuesta._

### Reto 2 — Ingresos por categoría con un umbral

Finanzas quiere saber qué categorías generaron al menos S/ 10 000 en ingresos entre agosto y septiembre. El umbral es una cifra redonda cercana a la cuarta parte del ingreso total de la base (que usted verificó en el ejercicio 9): una categoría por debajo de él aporta menos de una cuarta parte. El ingreso se calcula desde las líneas de detalle para no duplicar totales.

Se solicita:

1.  Ejecutar la celda, que contiene dos consultas: el ingreso de todas las categorías y el de las que superan el umbral con HAVING.

2.  Distinguir qué filtra HAVING y por qué no podría escribirse en WHERE.

3.  Interpretar qué categorías quedan dentro y fuera del umbral.

**Celda 29 de 30**

In [ ]:
consulta = """SELECT pr.categoria,
       COUNT(DISTINCT d.id_pedido) AS pedidos_con_la_categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
ORDER BY ingreso DESC, pr.categoria;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.categoria,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
display(pd.read_sql_query(consulta, conexion))

**Pregunta de cierre.** ¿Qué categorías superan el umbral de S/ 10 000 y cuál queda fuera? ¿Qué diferencia hay entre su número de unidades y su ingreso? ¿Qué advertencia haría por el periodo analizado?

**Respuesta:** _Escriba aquí su respuesta._

### Reto 3 — Productos con rotación inferior al promedio

Compras revisa qué productos rotaron menos que el promedio de unidades por producto, para decidir cuáles no reponer en la próxima compra. El promedio se calcula sobre los 14 productos del catálogo, incluidos los que no se vendieron, porque la pregunta es sobre todo el catálogo. La consulta usa la vista vista_ventas_producto de la celda 23 y una subconsulta.

Se solicita:

1.  Ejecutar la celda, que muestra el promedio de unidades por producto y los productos que están por debajo de él, consultando la vista.

2.  Identificar qué parte de la consulta es la subconsulta y qué valor devuelve.

3.  Interpretar el resultado con la cautela que exige un periodo de dos meses.

**Celda 30 de 30**

In [ ]:
consulta = """SELECT ROUND(AVG(unidades_vendidas), 2) AS promedio_unidades,
       COUNT(*) AS productos_del_catalogo
FROM vista_ventas_producto;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT id_producto, nombre_producto, categoria, unidades_vendidas
FROM vista_ventas_producto
WHERE unidades_vendidas < (SELECT AVG(unidades_vendidas)
                           FROM vista_ventas_producto)
ORDER BY unidades_vendidas, id_producto;"""
display(pd.read_sql_query(consulta, conexion))

**Pregunta de cierre.** ¿Cuántos productos están por debajo del promedio y cuáles no se vendieron? ¿Qué decisión de reposición sugiere el resultado y qué dato adicional pediría antes de dejar de reponer un producto?

**Respuesta:** _Escriba aquí su respuesta._

## Conclusiones

Escriba tres conclusiones técnicas y de negocio de este laboratorio.

1. _Escriba aquí su conclusión._

2. _Escriba aquí su conclusión._

3. _Escriba aquí su conclusión._